<a href="https://colab.research.google.com/github/ashutosh-kedar/RT-DETR-v2-Object-Detection-Fine-Tuning/blob/main/RT_DETR_POC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
try:
  import datasets
  import torchmetrics
  import pycocotools
except ModuleNotFoundError:
  !pip install datasets
  !pip install torchmetrics[detection]

import transformers
import numpy as np
import torch

### Loading the dataset

In [ ]:
from datasets import load_dataset

dataset = load_dataset('mrdbourke/trashify_manual_labelled_images')
dataset

In [ ]:
dataset.get('train')[0]

In [ ]:
from pprint import pprint

pprint(dataset.get('train')[0])

In [ ]:
dataset.get('train').features

In [ ]:
categories = dataset.get('train').features.get('annotations').get('category_id')
categories

In [ ]:
categories.feature.names

In [ ]:
label2id = {label:id for id,label in enumerate(categories.feature.names)}
id2label = {id:label for label,id in label2id.items()}
label2id,id2label

In [ ]:
colour_palette = {
    'bin': (0, 0, 224),         #  (red, green, blue)
    'not_bin': (255, 80, 80),
    'hand': (148, 0, 211),
    'not_hand': (255, 80, 80),
    'trash': (0, 255, 0),
    'not_trash': (255, 80, 80),
    'trash_arm': (255, 140, 0),
}

In [ ]:
sample_record = dataset.get('train')[0]

In [ ]:
import PIL
# we need this as image dimensions are too high
def half_image_dimensions(pil_image:PIL.Image):
  width, height = pil_image.size
  return pil_image.resize(size=(width//2,height//2))

In [ ]:
dataset.get('train')[0].get('image')

In [ ]:
half_image_dimensions(sample_record.get('image'))

In [ ]:
from typing import List

#when we halfed the image we also need to half the bounding box coordinated
def half_bbox(bbox):

  halfed_bbox = list()
  if isinstance(bbox, List):
    for box in bbox:
      if isinstance(box,List):
        halfed_bbox = [[index//2 for index in coordinates] for coordinates in bbox]
      else:
        halfed_bbox = [index//2 for index in bbox]

  if isinstance(bbox,torch.Tensor):
    return bbox // 2

  return halfed_bbox

In [ ]:
sample_bbox = sample_record.get('annotations').get('bbox')
sample_bbox

In [ ]:
half_bbox(sample_bbox)

### Out datset had bounding box format of 'xywh' x-cordinate y-cordinate h-heigh w-width

Ploting bounding boxes on random images from the datasets

In [ ]:
labels = categories.feature.names

In [ ]:
import random
from torchvision.ops import box_convert
import torch

from torchvision.transforms.functional import pil_to_tensor,to_pil_image
from torchvision.utils import draw_bounding_boxes

random_idx = random.randint(0,len(dataset.get('train')))


random_sample = dataset.get('train')[random_idx]

random_sampled_image = random_sample.get('image')
random_sampled_image_bbox = random_sample.get('annotations').get('bbox')

random_sampled_image = half_image_dimensions(random_sampled_image)
random_sampled_image_bbox = half_bbox(random_sampled_image_bbox)

random_sampled_image_bbox = torch.tensor(random_sampled_image_bbox)

converted_bbox = box_convert(boxes=random_sampled_image_bbox,in_fmt='xywh',out_fmt='xyxy')

random_sample_label_names = [labels[label] for label in random_sample['annotations'].get('category_id')]

random_sample_bbox_colours = [colour_palette[label] for label in random_sample_label_names]


random_sample_image_tensor = pil_to_tensor(random_sampled_image)


random_sample_image_bbox_tensor = draw_bounding_boxes(
    image=random_sample_image_tensor,
    boxes=converted_bbox,
    labels=random_sample_label_names,
    colors=random_sample_bbox_colours,
    width=3,
    font_size=25,
    fill_labels=True

)

random_sample_pil_image_bbox = to_pil_image(
    pic = random_sample_image_bbox_tensor
)

random_sample_pil_image_bbox

### Using real time detection using transformer model RT-DeTr v2 using ResNet 50 Architecture

In [ ]:
from transformers import AutoModelForObjectDetection

MODEL_NAME = 'PekingU/rtdetr_v2_r50vd'


model = AutoModelForObjectDetection.from_pretrained(
    pretrained_model_name_or_path=MODEL_NAME,
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True  # if this is false it gives error as no. of labels doesnt match the no. of label the model was trained on
)

model

In [ ]:
model.class_embed   # for predicting the label for the bounding box

In [ ]:
model.bbox_embed  # for predicting the 4 co-ordinated of the bounding box.

In [ ]:
def get_model_parameters(model):

  trainable_parameters = sum([parameter_tensor.numel() for parameter_tensor in model.parameters() if parameter_tensor.requires_grad==True])
  non_trainable_parameters = sum([parameter_tensor.numel() for parameter_tensor in model.parameters() if parameter_tensor.requires_grad==False])
  total_parameters = trainable_parameters + non_trainable_parameters

  print('Trainable Parameters:',trainable_parameters)
  print('Non-Trainable parameter:',non_trainable_parameters)
  print('Total Parameters:',total_parameters)

  return {
      'trainable_parameters':trainable_parameters,
      'non_trainable_parameters':non_trainable_parameters,
      'total_parameters':total_parameters
  }

In [ ]:
model_parameter_count = get_model_parameters(model)

In [ ]:
random_sample

In [ ]:
model.forward?

In [ ]:
# model(pixel_values=random_sample['image'])

In [ ]:
# Need to preprocess the image before passing it to the model.


from transformers import AutoImageProcessor

image_preprocessor = AutoImageProcessor.from_pretrained(
    pretrained_model_name_or_path=MODEL_NAME,
    use_fast=True,
    do_convert_annotations=True,
    size={
        "shortest_edge":640,
        "longest_edge": 640
    },
    do_pad=True
)

In [ ]:
# random_preprocessed_image = image_preprocessor.preprocess(
#     images=random_sample.get('image'),
#     annotations=random_sample.get('annotations')

# )

In [ ]:
from pprint import pprint

random_preprocessed_image = image_preprocessor.preprocess(
    images=random_sample.get('image'),
    annotations=None,
    return_tensors='pt'
)

random_preprocessed_image.keys()

In [ ]:
print('Original image dimensions:',random_sample.get('image').size)
print('Preprocessed image dimensions:',random_preprocessed_image.pixel_values.shape)

In [ ]:
with torch.inference_mode():
  model_result = model(**random_preprocessed_image)

pprint(model_result.keys())

In [ ]:
model_result.pred_boxes.shape #  [1, 300, 4] -> (batch_size,bounding_box_predicted,4_co-ordinates)

In [ ]:
model_result.logits.shape   # [1, 300, 7] -> (batch_size,bounding_box_predicted,logints_for_7_labels)

In [ ]:
type(model_result)

In [ ]:
model_result.pred_boxes[:,0,:]

In [ ]:
random_sample

In [ ]:
from dataclasses import dataclass,asdict
from typing import List

@dataclass
class COCOAnnotation:

  image_id: int
  category_id: int
  area: float
  bbox: List[float]
  iscrowd: int

  def __post_init__(self):
    pass # validation for the data can we given here. can raise value error.



@dataclass
class EntireCOCOAnnotation:

  image_id :int
  annotations: List[COCOAnnotation]


In [ ]:
random_sample

In [ ]:
from dataclasses import asdict

def tranform_into_coco_annotation(image_annotation):

  image_id = image_annotation.get('image_id')
  category_ids = image_annotation.get('annotations').get('category_id')
  bbox = image_annotation.get('annotations').get('bbox')
  is_crowded = image_annotation.get('annotations').get('iscrowd')
  area = image_annotation.get('annotations').get('area')

  annotations = [
      {
          'image_id': image_id,
          'category_id': category_id,
          'area': area,
          'bbox': single_bbox,
          'iscrowd': is_crowd
      } for category_id,area,single_bbox,is_crowd in zip(category_ids,area,bbox,is_crowded)
  ]

  return asdict(EntireCOCOAnnotation(
      image_id=image_id,
      annotations=annotations
  ))


In [ ]:
tranform_into_coco_annotation(random_sample)

In [ ]:
preprocessed_sample_image_with_annotation = image_preprocessor.preprocess(
    images=random_sample.get('image'),
    annotations=tranform_into_coco_annotation(random_sample)
)

In [ ]:
preprocessed_sample_image_with_annotation

In [ ]:
preprocessed_sample_image_with_annotation.keys()

In [ ]:
import torch


with torch.inference_mode():
  model_result = model(**preprocessed_sample_image_with_annotation)

model_result.keys()

In [ ]:
model_result.get('logits').shape

### Need to post process to convert logits to the probability values

In [ ]:
random_sample_post_process_result = image_preprocessor.post_process_object_detection(
    outputs=model_result,
    threshold=0.5,
    target_sizes = preprocessed_sample_image_with_annotation['labels'][0]["orig_size"].unsqueeze(0)
)

In [ ]:
random_sample_post_process_result[0].keys()

In [ ]:
random_sample_post_process_result[0].get('labels'),id2label

In [ ]:
type(random_sample.get('image'))

In [ ]:
from torchvision.transforms.functional import to_pil_image, pil_to_tensor
from torchvision.utils import draw_bounding_boxes
half_image = half_image_dimensions(random_sample.get('image'))
half_bbox_coordinates = half_bbox(random_sample_post_process_result[0].get('boxes'))
# half_bbox_coordinates[0],random_sample_post_process_result[0].get('boxes')[0]

labels = [id2label[label_id.item()] for label_id in random_sample_post_process_result[0].get('labels')]

colors = [colour_palette.get(a_label) for a_label in labels]

bbox_plotted_random_sampled_image = to_pil_image(
  pic=draw_bounding_boxes(
    image=pil_to_tensor(
        pic=half_image
    ),
    boxes=half_bbox_coordinates,
    labels=labels,
    colors=colors,
    # fill = True,
    width = 3
  )
)

bbox_plotted_random_sampled_image

This is models initial prediction without training. The pipeline is as follows.

1. Random sampled image first goes through pre processing
2. Then the pre processed image is passed to the model.
3. Once the model returns the result the returned result is post processed.
4. The post processed result containing the bbox are drawn on to the image

In [ ]:
len(half_bbox_coordinates) # total number of bbox above the threshold we have in the post process
# model predicts 300 bbox. Based on our threshold the 300 bbox are filtered. Each 300 bbox predicted has a score.

Doing post process of the model result.

In [ ]:
model_result

Doing the post process manually.

Getting the scores first.



In [ ]:
import torch
THRESHOLD=0.5

sigmoid_of_logits = torch.sigmoid(model_result.logits)
flattened_sigmoid_result = torch.flatten(sigmoid_of_logits,start_dim=1)

top_100_sigmoid_values,top_100_sigmoid_values_index = torch.topk(
    input=flattened_sigmoid_result,
    k=100,
    dim=-1
)

top_100_sigmoid_values_filter = top_100_sigmoid_values > THRESHOLD

sorted_top_100_filtered_sigmoid_values,sorted_top_100_filtered_sigmoid_values_index = torch.sort(
    input= top_100_sigmoid_values[top_100_sigmoid_values_filter],
    descending=True
)

# len(sorted_top_100_filtered_sigmoid_values)
sorted_top_100_filtered_sigmoid_values

In [ ]:
random_sample_post_process_result[0].get('scores')

In [ ]:
torch.isclose(input=sorted_top_100_filtered_sigmoid_values,
               other=random_sample_post_process_result[0].get('scores'))

In [ ]:
random_sample_post_process_result[0].get('boxes').shape

In [ ]:
model_result.pred_boxes.shape

In [ ]:
len(top_100_sigmoid_values_index[0])

In [ ]:
model_result.pred_boxes.shape

In [ ]:
bboc_top_100_score_index = top_100_sigmoid_values_index[0] // total_labels
bboc_top_100_score_index

In [ ]:
from  torchvision.ops import box_convert


total_labels= len(model.config.label2id)

bboc_top_100_score_index = top_100_sigmoid_values_index[0] // total_labels

top_100_bbox = model_result.pred_boxes[0][bboc_top_100_score_index]

top_100_sorted_bbox_with_threshold = top_100_bbox[sorted_top_100_filtered_sigmoid_values_index]

#model outputs cxcywh we need to convert it to xyxy

converted_bbox_formatted = box_convert(
    boxes=top_100_sorted_bbox_with_threshold,
    in_fmt='cxcywh',
    out_fmt='xyxy'
)

# The model normalizes the coordinated. So we need to de-normalize

original_size = preprocessed_sample_image_with_annotation.get('labels')[0]['orig_size']

original_size_tensor = torch.tensor(
    [
        original_size[1],
        original_size[0],
        original_size[1],
        original_size[0],
    ]
)
# converted_bbox_formatted.shape, original_size_tensor.shape
denormalized_converted_bbox = converted_bbox_formatted * original_size_tensor # this is normal multiplication and not mat mul

# pytorch internally convert 20,4 * 4
#to
#(20,4) * (20,4) expands the 4 to 20 rows ti make multiplcation

In [ ]:
original_size_tensor

In [ ]:
denormalized_converted_bbox.shape,random_sample_post_process_result[0].get('boxes').shape

In [ ]:
torch.isclose(input=denormalized_converted_bbox,other=random_sample_post_process_result[0].get('boxes'))

In [ ]:
random_sample_post_process_result[0].get('labels')

In [ ]:

total_labels= len(model.config.label2id)

top_100_label_ids = top_100_sigmoid_values_index % total_labels

top_100_label_ids = top_100_label_ids[top_100_sigmoid_values_filter]

top_100_label_ids

In [ ]:
torch.isclose(input=top_100_label_ids,other=random_sample_post_process_result[0].get('labels'))